# 03 — Random Forest baseline

Train a Random Forest on hand-crafted scalar features extracted from the global view. This is the natural baseline:

- **Cheap to train** — seconds, not minutes.
- **Interpretable** — SHAP shows which features matter.
- **Connects to DATA 305 Week 2** — bagging + random feature subsampling.

Run via the script (recommended) or step through here for exploration:

```bash
python scripts/train_model.py model=random_forest data=small
```

In [ ]:
import numpy as np
from omegaconf import OmegaConf

from exoplanet_hunter.eval.metrics import classification_metrics
from exoplanet_hunter.features import FEATURE_NAMES, extract_features
from exoplanet_hunter.models import build_random_forest
from exoplanet_hunter.training.data_module import load_views, train_val_test_split

# Load the processed dataset (run scripts/build_dataset.py first).
views = load_views("../data/processed/views.npz")
train_v, val_v, test_v = train_val_test_split(views, seed=42)
print(f"train={len(train_v.labels)}  val={len(val_v.labels)}  test={len(test_v.labels)}")

In [ ]:
X_train = np.array([extract_features(v) for v in train_v.global_views])
X_test  = np.array([extract_features(v) for v in test_v.global_views])
y_train = train_v.labels.astype(int)
y_test  = test_v.labels.astype(int)

cfg = OmegaConf.load("../conf/model/random_forest.yaml")
rf = build_random_forest(cfg)
rf.fit(X_train, y_train)
y_score = rf.predict_proba(X_test)[:, 1]
metrics = classification_metrics(y_test, y_score)
print(metrics)

In [ ]:
# Feature importance (Gini-based)
import matplotlib.pyplot as plt

imp = rf.named_steps['clf'].feature_importances_
order = np.argsort(imp)[::-1]
plt.figure(figsize=(8, 4))
plt.bar(range(len(imp)), imp[order])
plt.xticks(range(len(imp)), [FEATURE_NAMES[i] for i in order], rotation=60, ha="right")
plt.ylabel("importance"); plt.title("RF feature importance")
plt.tight_layout();